# HW6 — Smart Beta and Factor Investing

**FINM 36700: Portfolio and Risk Management**  
**Group Members: Ying Ding, Jiarui Feng, Hoang Pham, Nikita Rodin**

This notebook covers the graded parts of the case study [Smart Beta and Factor Investing](https://markhendricks.github.io/finm-portfolio/case_studies/Smart%20Beta%20and%20Factor%20Investing.html):

- **Section 2. The Factors.** Factor performance, correlations, and tangency portfolios.
- **Section 3. Testing Modern LPMs.** Time-series and cross-sectional tests of the specified factor models.

Section 1 is marked READING and is not included.

Data, from the course [Box folder](https://uchicago.app.box.com/s/w3oqz9iy0kav3n3l4pa04p6etndo2ez2):

- `factor_pricing_data.xlsx`: monthly excess returns for six factors and 49 industry portfolios, January 1980–April 2025 (544 months). The post-case comparison uses January 2015–April 2025.

**AI disclosure.** AI assistance was used to draft the content and supporting code.

## Data and methods

To rerun this notebook, place `factor_pricing_data.xlsx` next to the notebook. All calculation code and saved outputs are included below.

Returns and alphas are annualized by multiplying monthly means by 12. Volatility uses the sample standard deviation times $\sqrt{12}$; Sharpe is monthly mean divided by monthly volatility times $\sqrt{12}$. Monthly skewness, correlations, and $R^2$ are not annualized. All returns are decimal values. Reported performance is before trading costs, fees, and taxes.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

pd.set_option('display.max_rows', 60)
RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)

def load_returns(filename, sheet):
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f'Place {filename} next to this notebook.')
    data = pd.read_excel(path, sheet_name=sheet, index_col=0)
    data.index = pd.to_datetime(data.index).to_period('M')
    data.columns = data.columns.str.strip()
    data = data.sort_index().astype(float)
    assert data.index.is_unique
    assert data.index.equals(pd.period_range(data.index.min(), data.index.max(), freq='M'))
    assert np.isfinite(data.to_numpy()).all()
    return data

def summary(returns, benchmarks=None):
    if isinstance(returns, pd.Series):
        returns = returns.to_frame()
    vol = returns.std(ddof=1)
    result = pd.DataFrame({'Mean': returns.mean() * 12,
                           'Vol': vol * np.sqrt(12),
                           'Sharpe': returns.mean() / vol * np.sqrt(12),
                           'Skew': returns.skew()})
    if benchmarks is not None:
        assert returns.index.equals(benchmarks.index)
        for factor in ['MKT', 'HML']:
            result['Corr ' + factor] = returns.apply(lambda x: x.corr(benchmarks[factor]))
    return result

def show(frame, percent=()):
    # Format a copy, preserving the underlying decimal values.
    formatted = frame.copy()
    for col in formatted:
        if pd.api.types.is_numeric_dtype(formatted[col]):
            formatted[col] = formatted[col].map(
                lambda x: 'n.a.' if pd.isna(x) else (f'{x:.2%}' if col in percent else f'{x:.3f}'))
    display(formatted)


In [2]:
factor_names = ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']
factors = load_returns('factor_pricing_data.xlsx', 'factors (excess returns)')[factor_names]
industries = load_returns('factor_pricing_data.xlsx', 'portfolios (excess returns)')
assert factors.index.equals(industries.index)
assert industries.shape[1] == 49
print(f'Sample: {factors.index.min()} through {factors.index.max()}, {len(factors)} monthly observations.')
# Both sheets already contain excess returns: do not subtract RF again.


Sample: 1980-01 through 2025-04, 544 monthly observations.


## 2. The Factors

### 2.1. Factor summary statistics

**Table SB1. Full sample, January 1980-April 2025**

In [3]:
factor_full = summary(factors)[['Mean', 'Vol', 'Sharpe']]
show(factor_full, percent=['Mean', 'Vol'])

factor_full.to_csv(RESULTS / 'smart_beta_q2_1_factor_stats.csv')

,Mean,Vol,Sharpe
MKT,8.49%,15.64%,0.543
SMB,0.54%,10.14%,0.053
HML,2.65%,10.89%,0.243
RMW,4.51%,8.31%,0.543
CMA,2.77%,7.25%,0.382
UMD,6.18%,15.35%,0.402


### 2.2. Risk premia and performance since 2015

**Table SB2. January 2015-April 2025**

In [4]:
factor_recent = summary(factors.loc['2015':])[['Mean', 'Vol', 'Sharpe']]
show(factor_recent, percent=['Mean', 'Vol'])

factor_recent.to_csv(RESULTS / 'smart_beta_q2_2_post2015_stats.csv')

,Mean,Vol,Sharpe
MKT,10.74%,15.86%,0.677
SMB,-2.78%,10.35%,-0.269
HML,-1.56%,13.08%,-0.119
RMW,4.47%,7.29%,0.613
CMA,-1.30%,8.23%,-0.158
UMD,2.52%,13.84%,0.182


All six factors have positive full-sample average excess returns. MKT and RMW have the highest standalone Sharpe ratios, both approximately 0.54; SMB has a very small premium (0.54% annually) and a Sharpe of 0.05. Since 2015, MKT and RMW remain strong, while SMB, HML, and CMA have negative average returns. UMD remains positive but weakens to 2.52% annually and a Sharpe of 0.18. Positive historical averages are estimates of premia, not proof of positive population expected returns.

### 2.3. Correlation and redundancy

**Table SB3. Full-sample factor correlations**

In [5]:
show(factors.corr())
# A spanning regression supports the HML redundancy discussion.
hml_fit = sm.OLS(factors['HML'], sm.add_constant(factors[['MKT', 'SMB', 'RMW', 'CMA']])).fit()
print(f'HML annual residual alpha: {12 * hml_fit.params["const"]:.2%}')
print(f'HML spanning R-squared: {hml_fit.rsquared:.3f}')

factors.corr().to_csv(RESULTS / 'smart_beta_q2_3_correlations.csv')

,MKT,SMB,HML,RMW,CMA,UMD
MKT,1.000,0.228,-0.206,-0.250,-0.352,-0.179
SMB,0.228,1.000,-0.027,-0.413,-0.054,-0.058
HML,-0.206,-0.027,1.000,0.220,0.681,-0.214
RMW,-0.250,-0.413,0.220,1.000,0.140,0.074
CMA,-0.352,-0.054,0.681,0.140,1.000,0.001
UMD,-0.179,-0.058,-0.214,0.074,0.001,1.000


HML annual residual alpha: -1.57%
HML spanning R-squared: 0.487


Construction keeps many pairwise correlations modest, but does not make the factors orthogonal. HML-CMA is the largest positive correlation (0.681); SMB-RMW is -0.413. UMD has correlations of -0.179 with MKT and -0.214 with HML, making it a potentially useful complement.

HML overlaps substantially with CMA. A regression of HML on the other four FF5 factors gives R-squared 0.487 and annual alpha -1.57%. Thus HML is not an exact replication of the other factors, but its positive standalone mean does not translate into a positive residual premium. Its small six-factor tangency weight below provides additional evidence of limited incremental value in this sample. Pairwise correlation alone would not establish redundancy.

### 2.4. Tangency portfolios

For each specified factor set, weights are proportional to Sigma^(-1) mu and normalized to sum to one. Short positions are allowed. These weights combine factor return series; the sum-to-one convention does not describe the gross underlying stock exposure of the long-short factors.

**Table SB4. Unconstrained tangency weights**

In [6]:
def tangency_weights(returns):
    direction = np.linalg.solve(returns.cov(), returns.mean())
    weights = pd.Series(direction / direction.sum(), index=returns.columns)
    assert np.isclose(weights.sum(), 1)
    return weights

weights6 = tangency_weights(factors)
subset = ['MKT', 'SMB', 'HML', 'UMD']
weights4 = tangency_weights(factors[subset])
weights = pd.concat([weights6.rename('Six factors'), weights4.rename('Four factors')], axis=1).reindex(factor_names)
show(weights, percent=weights.columns)
portfolio_returns = pd.DataFrame({'Six factors': factors @ weights6,
                                 'Four factors': factors[subset] @ weights4})
show(summary(portfolio_returns)[['Mean', 'Vol', 'Sharpe']], percent=['Mean', 'Vol'])

weights.to_csv(RESULTS / 'smart_beta_q2_4_tangency_weights.csv')
summary(portfolio_returns)[['Mean', 'Vol', 'Sharpe']].to_csv(RESULTS / 'smart_beta_q2_4_portfolio_stats.csv')

,Six factors,Four factors
MKT,21.61%,37.15%
SMB,6.67%,-5.37%
HML,-1.97%,36.79%
RMW,30.59%,n.a.
CMA,31.66%,n.a.
UMD,11.44%,31.43%


,Mean,Vol,Sharpe
Six factors,4.78%,4.00%,1.194
Four factors,6.04%,6.67%,0.906


RMW and CMA receive the largest six-factor weights (30.59% and 31.66%). MKT receives 21.61%, UMD 11.44%, and SMB 6.67%. HML is a small short position (-1.97%), so it contributes little incremental value once profitability and investment are available. SMB remains useful despite its low standalone mean, partly because of its negative covariance with RMW.

With only MKT, SMB, HML, and UMD, HML rises to 36.79%, MKT to 37.15%, and UMD to 31.43%; SMB becomes a -5.37% short. HML therefore captures useful exposure when RMW and CMA are absent. The annualized sample Sharpe is 1.194 for six factors versus 0.906 for four. Style importance depends on the available opportunity set and covariance structure, not just standalone mean returns. These are in-sample estimates and may be unstable out of sample.

## 3. Testing Modern Linear Pricing Models

### 3.1. AQR time-series regressions

For each industry, estimate r_i,t(excess) = alpha_i + beta_i,MKT MKT_t + beta_i,HML HML_t + beta_i,RMW RMW_t + beta_i,UMD UMD_t + error_i,t by OLS with an intercept. Table SB5 reports all 49 estimated annual alphas and time-series R-squared values.

**Table SB5. AQR model, January 1980-April 2025**

In [7]:
models = {'CAPM': ['MKT'],
          'FF3': ['MKT', 'SMB', 'HML'],
          'FF5': ['MKT', 'SMB', 'HML', 'RMW', 'CMA'],
          'AQR': ['MKT', 'HML', 'RMW', 'UMD']}
fits = {}
industry_results = {}
for model, columns in models.items():
    x = sm.add_constant(factors[columns])
    fits[model] = {asset: sm.OLS(industries[asset], x).fit() for asset in industries}
    industry_results[model] = pd.DataFrame({asset: {'Alpha': fit.params['const'] * 12,
                                                  'R-squared': fit.rsquared}
                                            for asset, fit in fits[model].items()}).T
show(industry_results['AQR'], percent=['Alpha'])

industry_results['AQR'].to_csv(RESULTS / 'smart_beta_q3_1_aqr_industries.csv')

,Alpha,R-squared
Agric,1.10%,0.340
Food,0.42%,0.461
Soda,1.82%,0.304
Beer,1.11%,0.421
Smoke,4.30%,0.266
Toys,-3.46%,0.509
Fun,4.13%,0.610
Books,-3.54%,0.690
Hshld,-1.02%,0.559
Clths,-2.29%,0.618


AQR alphas range from -5.42% for RlEst to +6.86% for Softw. Its average R-squared is 0.574, but industry coverage differs greatly: Gold is only 0.049 and BusSv is 0.849. The four factors capture a substantial part of common industry variation while leaving economically meaningful average-return errors.

### 3.2. Alpha MAE and comparison of the four models

The assignment repeats the number 2; this section answers both the AQR MAE question and the subsequent CAPM/FF3/FF5 comparison. MAE = (1/49) sum_i |annual alpha_i|. The true alphas should be zero under a correct traded-factor pricing model, with nonzero estimated values possible from sampling error.

**Table SB6. Time-series pricing and decomposition**

In [8]:
model_summary = pd.DataFrame({model: {'TS MAE': result['Alpha'].abs().mean(),
                                           'Mean TS R2': result['R-squared'].mean()}
                              for model, result in industry_results.items()}).T
show(model_summary, percent=['TS MAE'])

model_summary.to_csv(RESULTS / 'smart_beta_q3_2_model_stats.csv')

,TS MAE,Mean TS R2
CAPM,2.14%,0.524
FF3,2.45%,0.570
FF5,3.15%,0.594
AQR,2.46%,0.574


The AQR model has MAE 2.46% annually, which is economically meaningful and does not provide strong support for exact pricing. CAPM has the smallest MAE (2.14%), followed by FF3 (2.45%), AQR (2.46%), and FF5 (3.15%). Thus CAPM fits best under the requested time-series pricing-error criterion. Adding factors can raise R-squared while worsening alpha MAE: OLS minimizes squared time-series residuals, not absolute intercepts. This is a descriptive comparison, not a formal statistical rejection test.

### 3.3. Which factors matter? Should momentum be included?

MKT alone captures much of the common variation and delivers the smallest alpha MAE here. The additional factors improve decomposition but do not improve time-series pricing MAE over CAPM. HML has limited incremental portfolio value in the six-factor set, as discussed in Section 2. AQR and FF5 are not nested, so their difference cannot be attributed solely to momentum.

In [9]:
# Hold the FF5 factor set fixed and add UMD to assess its incremental role.
ff5_umd = {asset: sm.OLS(industries[asset], sm.add_constant(factors)).fit()
           for asset in industries}
mae_ff5_umd = np.mean([abs(fit.params['const']) * 12 for fit in ff5_umd.values()])
r2_ff5_umd = np.mean([fit.rsquared for fit in ff5_umd.values()])
print(f'FF5 + UMD: annual alpha MAE = {mae_ff5_umd:.2%}; average R-squared = {r2_ff5_umd:.3f}')

FF5 + UMD: annual alpha MAE = 2.87%; average R-squared = 0.599


To isolate the addition of UMD, adding it to FF5 reduces MAE from 3.15% to 2.87% and raises mean R-squared from 0.594 to 0.599. This is a modest improvement, and pricing MAE remains above CAPM. Momentum merits consideration because it offers distinct covariance exposure and historically positive returns, but these industry tests alone do not establish that it must be added to FF5. Evidence across other test assets and net implementation costs would be needed for a stronger recommendation.

### 3.4. Average time-series R-squared

The final column of Table SB6 reports the average across all 49 regressions for all four specified models. FF5 has the highest average R-squared (0.594), followed by AQR (0.574), FF3 (0.570), and CAPM (0.524). Explaining roughly 52%-59% of industry return variation makes the factors useful for a Linear Factor Decomposition, although they leave substantial residual variation. The highest R-squared model is not the best pricing model under alpha MAE.

### 3.5. Cross-sectional tests

For each model, first estimate industry betas using the time-series regressions above. Then estimate mean_i(annual excess return) = eta + beta_i transpose lambda + residual_i across the 49 industries. An intercept is included, following the group HW4 convention. Lambda and eta are annualized. The theoretical traded-factor restrictions are eta = 0 and lambda equal to the annualized factor means.

**Table SB7. Historical means versus cross-sectional premia**

In [10]:
premium_rows = []
cs_results = {}
cs_residuals = {}
for model, columns in models.items():
    beta = pd.DataFrame({asset: fit.params[columns] for asset, fit in fits[model].items()}).T
    cs = sm.OLS(industries.mean() * 12, sm.add_constant(beta)).fit()
    cs_results[model] = cs
    cs_residuals[model] = cs.resid
    for factor in columns:
        premium_rows.append({'Model': model, 'Factor': factor,
                             'TS premium': factors[factor].mean() * 12,
                             'CS premium': cs.params[factor]})
premia = pd.DataFrame(premium_rows).set_index(['Model', 'Factor'])
show(premia, percent=['TS premium', 'CS premium'])

premia.to_csv(RESULTS / 'smart_beta_q3_5_factor_premia.csv')

TS premium CS premium
Model Factor                      
CAPM  MKT         8.49%      0.21%
FF3   MKT         8.49%      3.76%
      SMB         0.54%     -4.47%
      HML         2.65%     -1.99%
FF5   MKT         8.49%      3.84%
      SMB         0.54%     -4.70%
      HML         2.65%     -2.58%
      RMW         4.51%      2.31%
      CMA         2.77%     -2.43%
AQR   MKT         8.49%      1.44%
      HML         2.65%     -3.26%
      RMW         4.51%      2.13%
      UMD         6.18%      0.19%

**Table SB8. Cross-sectional errors and comparison**

In [11]:
comparison = model_summary.copy()
for model, fit in cs_results.items():
    comparison.loc[model, 'CS MAE'] = fit.resid.abs().mean()
    comparison.loc[model, 'CS intercept'] = fit.params['const']
    comparison.loc[model, 'CS R2'] = fit.rsquared
show(comparison[['TS MAE', 'CS MAE', 'CS intercept', 'CS R2']],
     percent=['TS MAE', 'CS MAE', 'CS intercept'])

comparison.to_csv(RESULTS / 'smart_beta_q3_5_model_comparison.csv')

,TS MAE,CS MAE,CS intercept,CS R2
CAPM,2.14%,1.55%,8.69%,0.001
FF3,2.45%,1.20%,6.24%,0.379
FF5,3.15%,1.19%,6.00%,0.416
AQR,2.46%,1.38%,7.58%,0.210


The premia differ substantially. For CAPM, the cross-sectional market premium is only 0.21%, versus a historical 8.49%; its intercept is 8.69% and its cross-sectional R-squared is almost zero. SMB and HML change sign in FF3 and FF5. In AQR, the momentum premium falls from a historical 6.18% to 0.19%. The data therefore do not closely match the traded-factor restrictions.

Cross-sectional residual MAE is smaller than time-series alpha MAE for every model. FF5 is best by cross-sectional MAE (1.19%), narrowly ahead of FF3 (1.20%); AQR is 1.38% and CAPM is 1.55%. Unlike the time-series test, this regression freely estimates factor prices and an intercept. The lower errors do not validate the restriction that traded-factor premia equal their historical means. OLS minimizes squared cross-sectional errors, so it does not guarantee lower absolute-error MAE in general. The intercepts of 6.00%-8.69% are economically large, and the estimated betas themselves contain sampling error.

## Sources

[Smart Beta assignment](https://markhendricks.github.io/finm-portfolio/case_studies/Smart%20Beta%20and%20Factor%20Investing.html). Data: `factor_pricing_data.xlsx`, downloaded from the [course Box folder](https://uchicago.app.box.com/s/w3oqz9iy0kav3n3l4pa04p6etndo2ez2) on November 3, 2026. Cross-sectional regressions include an intercept, following the group HW4 convention.